# SPOT — worked examples

Run setup, then any case below. Each case creates a fresh configuration and uses the same production engine as the GUI and validation adapter. `QUALITY = 'full'` preserves MATLAB's 1024 × 1024 grid, supplied 648,000-row CSV catalogue, IAU2006 frame model and Poisson noise. `preview` is an explicit, different acquisition (256 × 256, synthetic catalogue, approximate frames, no Poisson noise).

Cases 1–7 correspond to the seven MATLAB example launchers. Additional presets and trajectory representations are shown at the end. Run All executes the seven public cases, including the ten-sample dataset; this can take a while. No results or figures are saved until their switches below are enabled.

In [ ]:
from pathlib import Path
from copy import deepcopy
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'ssa_emulator').is_dir():
    raise RuntimeError('Start Jupyter in the python folder containing ssa_emulator.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from ssa_emulator import run, convergence
from ssa_emulator.config import default_config, NAMES
from ssa_emulator.examples import example_config, run_example, batch, dataset_configurations
from ssa_emulator.plotting import plot_product

QUALITY = 'full'
SHOW_PLOTS = True
SAVE_PLOTS = False
SAVE_RESULTS = False
PLOT_PRODUCTS = ['camera', 'detector', 'dataset']  # also: radec, stages
PLOT_PHOTOMETRY = False
OUTPUT_DIR = ROOT / 'Outputs' / 'examples'

def setup_case(name):
    c = example_config(name, QUALITY)
    c.Output.Directory = str(OUTPUT_DIR)
    c.Output.SaveResults = SAVE_RESULTS
    c.Output.Plot = SHOW_PLOTS
    c.Plot.Save = SAVE_PLOTS
    c.Plot.Products = PLOT_PRODUCTS.copy()
    c.Plot.Photometry = PLOT_PHOTOMETRY
    return c

def show_summary(out):
    rows = []
    for e in out.Exposures:
        for t in e.Targets:
            rows.append(dict(exposure=e.Index, target=t.ID, appearance=t.Appearance,
                             visible=t.AnyVisible, annotated=t.HasAnnotation,
                             smear_pix=t.Morphology.EquivalentSmear_pix,
                             integrated_optical_signal=t.IntegratedOpticalSignal))
    display(pd.DataFrame(rows))
    print('Regimes:', [e.Truth.GeometryRegime for e in out.Exposures])
    print('Exposure / cadence:', out.Configuration.Time.Exposure_s,
          out.Configuration.Time.Cadence_s, 's')
    return out

## 1. Ground sky

Default: 1.5 s exposure and cadence, four physically propagated seeded targets. Edit exposure, pointing, or any target override before running. Empty `[]` overrides retain the configured values. Angle to LOS: 0° receding, 90° transverse, 180° approaching. Labels follow integrated brightness-weighted motion; they never switch renderers.

In [ ]:
c = setup_case('ground')
# EDIT HERE (optional):
# c.Time.Exposure_s = 0.5
# c.Time.Cadence_s = 0.5
# c.ExampleTarget.AngleToLOS_deg = [90, 0.2, 90, 0.2]
# c.ExampleTarget.RelativeSpeed_mps = 7500
# c.ExampleTarget.Direction_deg = [-35, 0, 35, -20]
# c.ExampleTarget.InitialAngles_deg = [[-6, 5], [-2, -4], [3, 2], [7, -6]]
# c.ExampleTarget.Range_m = 1e6
# c.ExamplePointing.AzEl_deg = [180, 60]
# c.ExamplePointing.Roll_deg = 0
out = show_summary(run_example(c, progress=print))

## 2. Earth background

30° off nadir; four targets against the Earth disk.

In [ ]:
c = setup_case('space_earth')
# c.Time.Exposure_s = 1.5
# c.ExamplePointing.OffNadir_deg = 30
out = show_summary(run_example(c, progress=print))

## 3. Earth limb

Sunlit scene at 70° off nadir, 2026-05-01 12:00 UTC, with **1 s exposure / 0.8 s cadence**, a 200,000-star catalogue cap, and explicit `legacy_gain` calibration of 1e6 electrons per model unit. Targets use two-body propagation, dynamic camera integration, and surface BRDF photometry. The detector plot displays raw DN with 1st–99.7th percentile contrast limits and a DN colorbar. Use `QUALITY='full'` for this configuration.


In [ ]:
c = setup_case('space_limb')
# c.Time.Exposure_s = 1.0  # exposure duration
# c.ExamplePointing.OffNadir_deg = 70
out = show_summary(run_example(c, progress=print))

## 4. Space sky

115° off nadir; sky-facing space observer.

In [ ]:
c = setup_case('space_sky')
# c.Time.Exposure_s = 1.5
# c.ExamplePointing.OffNadir_deg = 115
out = show_summary(run_example(c, progress=print))

## 5. Multiple exposures

Three 1.5 s exposures starting 1 s apart, so they overlap by 0.5 s. All exposures are plotted.

In [ ]:
c = setup_case('multi_exposure')
# c.Time.Exposure_s = 1.5
# c.Time.NumExposures = 3
out = show_summary(run_example(c, progress=print))

## 6. Dataset generation

MATLAB's `current_ground` base: 1 s exposure, 0.8 s cadence, 20 targets per sample, ten samples. Target rows are selected without replacement with seeds 101–110; detector seeds are 1–10. Random selections are repeatable within Python but need not match MATLAB's RNG realization. This cell retains only the index, not all acquisitions. Enable `SAVE_RESULTS` above to export data. Plots are off for this batch unless explicitly enabled here.

In [ ]:
c = setup_case('dataset')
c.Output.Plot = False
c.Plot.Save = False
NUM_SAMPLES = 10
FIRST_SELECTION_SEED = 100
# c.Input.NumTargets = 20
dataset_index = batch(dataset_configurations(c, NUM_SAMPLES, FIRST_SELECTION_SEED), progress=print)
display(dataset_index)
if c.Output.SaveResults:
    from datetime import datetime, timezone
    index_path = OUTPUT_DIR / ('dataset_index_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f') + '.csv')
    index_path.parent.mkdir(parents=True, exist_ok=True)
    dataset_index.to_csv(index_path, index=False)
    print(index_path)

## 7. Sensor and target CSV pools

MATLAB's ordered example: `current_ground`, 1 s exposure/cadence, sensor pool enabled and five targets starting at row 10. Explicit overrides apply **after** the selected pool rows. Seed overrides must remain empty for Keplerian, ephemeris or TLE targets.

In [ ]:
c = setup_case('csv_pools')
# c.Input.TargetSelection.Mode = 'random'
# c.Input.TargetSelection.Seed = 100
# For named rows:
# c.Input.NumTargets = 2
# c.Input.TargetSelection.Mode = 'ids'
# c.Input.TargetSelection.IDs = ['SEED0003', 'SEED0001']
# c.ExampleTarget.AngleToLOS_deg = [90, 0.2, 90, 0.2, 90]
# c.ExamplePointing.Roll_deg = 17
out = show_summary(run_example(c, progress=print))
print('Selected one-based rows:', out.Configuration.Input.SelectedTargetRows)

## 8. Other presets and trajectory inputs (optional)

Available MATLAB presets are listed below, including mixed appearances, figure8, tracking, Cartesian/geodetic/ECEF inputs, ephemerides, and LEO/deep-space TLEs. These share the renderer; differences are configuration only. The supplied sampled trajectory fixtures have finite time coverage. The optional trajectory run explicitly uses 0.2 s so it stays inside the input time window; extend the ephemeris files before longer exposures.

In [ ]:
print('\n'.join(NAMES))
RUN_EXTRA = False
EXTRA_CASE = 'trajectory_ephemeris_csv'
if RUN_EXTRA:
    c = setup_case(EXTRA_CASE)
    if EXTRA_CASE.startswith('trajectory_'):
        c.Time.Exposure_s = c.Time.Cadence_s = 0.2
    out = show_summary(run_example(c, progress=print))

## 9. Inspect an acquisition or check convergence (optional)

`out` is the last individual acquisition. Raw DN, rates, optical stages, detector stages, per-target histories and masks are available without rerunning. Python exposure indexing is zero-based; target pixel centres and boxes follow MATLAB's one-based convention. Convergence reruns the model with tighter sampling and can be expensive.

In [ ]:
e = out.Exposures[0]
print('Raw detector:', e.Image_DN.shape, e.Image_DN.dtype)
print('Masks:', e.Labels.Binary.shape, e.Labels.Instance.dtype, e.Labels.Semantic.dtype)
print('Target samples:', e.Sampling.TargetSampleCount)
# fig = plot_product(out, 'stages', exposure=0)
# fig = plot_product(out, 'photometry', exposure=0)
RUN_CONVERGENCE = False
if RUN_CONVERGENCE:
    # Use the fresh input c, not out.Configuration (whose seeds are resolved states).
    display(pd.DataFrame(convergence(c)))